In [29]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalizedQuantileTimeSeriesRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

In [30]:
# Generate synthetic panel data for five time series
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# LightGBM

In [31]:

#def create_mlforecast_multiquantile():
#        import lightgbm as lgb
#
#     # Define 90% and 50% prediction interval quantile LightGBM models
#     models = {
#         "LGBM-lo-90": lgb.LGBMRegressor(objective="quantile", alpha=0.05, random_state=42),
#         "LGBM-hi-90": lgb.LGBMRegressor(objective="quantile", alpha=0.95, random_state=42),
#         "LGBM-lo-50": lgb.LGBMRegressor(objective="quantile", alpha=0.25, random_state=42),
#         "LGBM-hi-50": lgb.LGBMRegressor(objective="quantile", alpha=0.75, random_state=42),
#     }
#     return MLForecast(
#         models=models,
#         freq="MS",
#         lags=[1, 7],
#     )
#models = create_mlforecast_multiquantile()

# RandomForestQuantileRegressor

In [ ]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

# RandomForestQuantileRegressor & HistGradientBoostingRegressor

In [37]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
                    quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "HGB-lo-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.05, max_iter=100, random_state=42),
        "HGB-hi-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.95, max_iter=100, random_state=42),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )

models = model_callable()

In [39]:
cqr = ConformalizedQuantileTimeSeriesRegressor(
     learner=models,
     intervals=[
         ("RF-lo-90", "RF-hi-90"),
         ("HGB-lo-90", "HGB-hi-90"),
     ],
 )
cqr.fit(train, horizon=12, n_windows=10, nexcp=False, weighted_refit=False, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,intervals,"[('RF-lo-90', ...), ('HGB-lo-90', ...)]"


In [40]:
cqr.predict_interval(h=12)

,unique_id,ds,RF-lo-90,RF-hi-90,HGB-lo-90,HGB-hi-90,RF-lo-90-cqr,RF-hi-90-cqr,HGB-lo-90-cqr,HGB-hi-90-cqr
0,1,1960-01-01,337.00,436.5,351.696226,548.200517,331.30,442.20,359.896226,540.000517
1,1,1960-02-01,305.00,548.0,280.350243,548.200517,296.00,557.00,293.350243,535.200517
2,1,1960-03-01,301.00,559.0,228.506346,548.200517,259.00,601.00,220.506346,556.200517
3,1,1960-04-01,301.00,559.0,217.483443,548.200517,260.00,600.00,214.345061,551.338899
4,1,1960-05-01,300.15,559.0,217.676035,548.200517,264.15,595.00,215.676035,550.200517
5,1,1960-06-01,277.00,559.0,217.676035,548.200517,250.85,585.15,191.070682,574.805870
6,1,1960-07-01,276.10,559.0,217.676035,548.200517,205.10,630.00,146.690066,619.186486
7,1,1960-08-01,277.00,559.0,218.292664,548.200517,219.00,617.00,150.488007,616.005174
8,1,1960-09-01,233.00,559.0,215.856502,548.200517,214.00,578.00,196.870533,567.186486
9,1,1960-10-01,211.00,559.0,207.942414,548.200517,219.00,551.00,209.942414,546.200517


In [41]:
cqr.evaluate(test, h=12)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RF,90%,0.1,0.833,282.354,465.688
1,RF-cqr,90%,0.1,1.000,329.996,329.996
2,HGB,90%,0.1,0.833,321.509,540.841
3,HGB-cqr,90%,0.1,0.917,349.529,354.218
